# Dataset Merging Compatibility Analysis

This notebook systematically explores the compatibility of merging pollutant, cause of death, and population datasets. The goal is to ensure that these datasets can be reliably combined for downstream analysis by aligning their country and year information, resolving naming inconsistencies, and verifying structural compatibility.

## 1. Importing Required Libraries

Import the necessary libraries for data manipulation and analysis.

In [1]:
import pandas as pd

## 2. Loading Datasets

Load the pollutant, cause of death, and population datasets into pandas DataFrames for further analysis.

In [2]:
all_cause_of_death_data = pd.read_csv('all_cause_of_death_data.csv')
merged_pollutants_data = pd.read_csv('merged_pollutants_data.csv')
population_data = pd.read_csv('processed_population_data.csv')

## 3. Initial Data Inspection

Inspect the first few rows and columns of the loaded datasets to understand their structure and contents.

In [3]:
# Ensure both column lists are of the same length
max_length = max(len(all_cause_of_death_data.columns), len(merged_pollutants_data.columns), len(population_data.columns))
all_cause_of_death_data_columns = list(all_cause_of_death_data.columns) + [''] * (max_length - len(all_cause_of_death_data.columns))
merged_pollutants_data_columns = list(merged_pollutants_data.columns) + [''] * (max_length - len(merged_pollutants_data.columns))
population_data_columns = list(population_data.columns) + [''] * (max_length - len(population_data.columns))

columns_df = pd.DataFrame({
    'all_cause_of_death_data_columns': all_cause_of_death_data_columns,
    'merged_pollutants_data_columns': merged_pollutants_data_columns,
    'population_data_columns': population_data_columns
})
columns_df

,all_cause_of_death_data_columns,merged_pollutants_data_columns,population_data_columns
0,Country,Country,Country
1,Cause,Region Name,Year
2,Year,Exposure Lower HAP,Population
3,Deaths,Exposure Mean HAP,
4,Deaths Upper,Exposure Upper HAP,
5,Deaths Lower,Year,
6,,Exposure Lower NO2,
7,,Exposure Mean NO2,
8,,Exposure Upper NO2,
9,,Exposure Lower OZONE,


In [4]:
summary = all_cause_of_death_data.describe(include='all').transpose()
summary['unique'] = all_cause_of_death_data.nunique()
summary['total_rows'] = len(all_cause_of_death_data)
summary

,count,unique,top,freq,mean,std,min,25%,50%,75%,max,total_rows
Country,1460844,204,Argentina,7161,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1460844
Cause,1460844,231,Chlamydial infection,6324,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1460844
Year,1460844.0,31,NaN,NaN,2005.0,8.944275,1990.0,1997.0,2005.0,2013.0,2020.0,1460844
Deaths,1460844.0,1391796,NaN,NaN,1282.73983,18964.96353,0.0,0.793357,17.172212,176.580968,2741251.777186,1460844
Deaths Upper,1460844.0,1391877,NaN,NaN,1580.191691,21899.554182,0.0,1.193869,24.52413,239.742295,3233138.518082,1460844
Deaths Lower,1460844.0,1391696,NaN,NaN,1044.265199,16689.528735,0.0,0.447873,10.941396,123.975273,2319056.920269,1460844


In [5]:
pollutants_summary = merged_pollutants_data.describe(include='all').transpose()
pollutants_summary['unique'] = merged_pollutants_data.nunique()
pollutants_summary['total_rows'] = len(merged_pollutants_data)
pollutants_summary

,count,unique,top,freq,mean,std,min,25%,50%,75%,max,total_rows
Country,6324,204,Afghanistan,31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6324
Region Name,6324,21,Western Europe,744,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6324
Exposure Lower HAP,6324.0,101,NaN,NaN,0.339231,0.368698,0.0,0.01,0.15,0.69,1.0,6324
Exposure Mean HAP,6324.0,101,NaN,NaN,0.360972,0.37221,0.0,0.02,0.19,0.73,1.0,6324
Exposure Upper HAP,6324.0,101,NaN,NaN,0.383831,0.375031,0.0,0.03,0.23,0.77,1.0,6324
Year,6324.0,31,NaN,NaN,2005.0,8.944979,1990.0,1997.0,2005.0,2013.0,2020.0,6324
Exposure Lower NO2,6324.0,2157,NaN,NaN,-0.120224,3.523401,-6.89,-2.64,-0.839,1.5225,16.9,6324
Exposure Mean NO2,6324.0,1382,NaN,NaN,10.075676,7.299635,-0.436,4.49,8.96,13.9,42.3,6324
Exposure Upper NO2,6324.0,699,NaN,NaN,11.007079,4.408951,4.35,7.36,10.6,13.7,28.3,6324
Exposure Lower OZONE,6324.0,615,NaN,NaN,37.623251,11.156769,-2.64,31.1,37.9,46.0,67.1,6324


In [6]:
population_summary = population_data.describe()
population_summary['total_rows'] = len(population_data)
population_summary

,Year,Population,total_rows
count,8215.000000,8.215000e+03,8215
mean,2005.000000,2.633952e+08,8215
std,8.944816,8.331187e+08,8215
min,1990.000000,8.798000e+03,8215
25%,1997.000000,1.334854e+06,8215
50%,2005.000000,8.720283e+06,8215
75%,2013.000000,5.689856e+07,8215
max,2020.000000,7.856139e+09,8215


## 4. Comparing Country Coverage Across Datasets

Compare the list of countries present in each dataset to identify overlaps, discrepancies, and unique entries. This helps ensure that merging will be meaningful and highlights any country naming inconsistencies.

In [7]:
from collections import defaultdict

def partition_by_membership(named_sets: dict[str, set]) -> dict[frozenset[str], set]:
    """
    Map each exact membership signature (the set of dataset names an item belongs to)
    to the items that have exactly that signature.
    """
    regions = defaultdict(set)
    if not named_sets:
        return regions

    all_items = set().union(*named_sets.values())
    for item in all_items:
        owners = frozenset(name for name, s in named_sets.items() if item in s)
        regions[owners].add(item)
    return regions

def _label_for_region(region_keys: frozenset[str], all_names: list[str]) -> str:
    names = sorted(region_keys)
    if len(names) == 0:
        return "In none (shouldn't happen if you union actual sets)"
    if len(names) == 1:
        return f"Unique in {names[0]}"
    if len(names) == len(all_names):
        return "Common in all of them"
    return "Common in " + " and ".join(names)

def report_regions(named_sets: dict[str, set], sort_values: bool = True) -> None:
    """
    Print each region with a human label, the count, and the values.
    Regions are shown in descending order by how many datasets they intersect.
    """
    if not named_sets:
        print("No datasets provided.")
        return

    all_names = sorted(named_sets.keys())
    regions = partition_by_membership(named_sets)

    # Sort regions: first by size of intersection (desc), then by name tuple (asc)
    def _region_sort_key(keys: frozenset[str]):
        return (-len(keys), tuple(sorted(keys)))

    for keys in sorted(regions.keys(), key=_region_sort_key):
        values = sorted(regions[keys]) if sort_values else list(regions[keys])
        label = _label_for_region(keys, all_names)
        print(f"{label}: {len(values)}")
        print(values)
        print("-" * 60)


pollutants_countries = set(merged_pollutants_data['Country'].unique())
death_countries = set(all_cause_of_death_data['Country'].unique())
population_countries = set(population_data['Country'].unique()) 

# If/when you add more datasets, just add more entries to this dict.
# e.g., population_countries = set(population_df['Country'].unique())
datasets = {
    "Pollutants": pollutants_countries,
    "Death": death_countries,
    "Population": population_countries,
}

report_regions(datasets)

Common in all of them: 175
['Afghanistan', 'Albania', 'Algeria', 'American Samoa', 'Andorra', 'Angola', 'Antigua and Barbuda', 'Argentina', 'Armenia', 'Australia', 'Austria', 'Azerbaijan', 'Bahrain', 'Bangladesh', 'Barbados', 'Belarus', 'Belgium', 'Belize', 'Benin', 'Bermuda', 'Bhutan', 'Bosnia and Herzegovina', 'Botswana', 'Brazil', 'Brunei Darussalam', 'Bulgaria', 'Burkina Faso', 'Burundi', 'Cabo Verde', 'Cambodia', 'Cameroon', 'Canada', 'Central African Republic', 'Chad', 'Chile', 'China', 'Colombia', 'Comoros', 'Costa Rica', 'Croatia', 'Cuba', 'Cyprus', 'Czechia', 'Denmark', 'Djibouti', 'Dominica', 'Dominican Republic', 'Ecuador', 'El Salvador', 'Equatorial Guinea', 'Eritrea', 'Estonia', 'Eswatini', 'Ethiopia', 'Fiji', 'Finland', 'France', 'Gabon', 'Georgia', 'Germany', 'Ghana', 'Greece', 'Greenland', 'Grenada', 'Guam', 'Guatemala', 'Guinea', 'Guinea-Bissau', 'Guyana', 'Haiti', 'Honduras', 'Hungary', 'Iceland', 'India', 'Indonesia', 'Iraq', 'Ireland', 'Israel', 'Italy', 'Jamaica', 

### 4.1 Harmonizing Country Names

Standardize country names across datasets to ensure consistency and enable accurate merging.

In [8]:
# Rename country names in all_cause_of_death_data to match pollutants data
all_cause_of_death_data['Country'] = all_cause_of_death_data['Country'].replace({
    'Taiwan': 'Taiwan (Province of China)',
    'Türkiye': 'Turkey'
})

# Verify the changes
cause_of_death_locations = set(all_cause_of_death_data['Country'].unique())
common_countries = pollutants_countries.intersection(cause_of_death_locations)
only_in_pollutants = pollutants_countries - cause_of_death_locations
only_in_cause_of_death = cause_of_death_locations - pollutants_countries

print("Common countries:", len(common_countries))
print("Only in pollutants data:", only_in_pollutants)
print("Only in cause of death data:", only_in_cause_of_death)

Common countries: 204
Only in pollutants data: set()
Only in cause of death data: set()


In [9]:
population_name_fixes = {
    'Bahamas, The': 'Bahamas',
    'Bolivia': 'Bolivia (Plurinational State of)',
    'Congo, Dem. Rep.': 'Democratic Republic of the Congo',
    'Congo, Rep.': 'Congo',
    "Cote d'Ivoire": "Côte d'Ivoire",
    'Egypt, Arab Rep.': 'Egypt',
    'Gambia, The': 'Gambia',
    'Iran, Islamic Rep.': 'Iran (Islamic Republic of)',
    "Korea, Dem. People's Rep.": "Democratic People's Republic of Korea",
    'Korea, Rep.': 'Republic of Korea',
    'Kyrgyz Republic': 'Kyrgyzstan',
    'Lao PDR': "Lao People's Democratic Republic",
    'Micronesia, Fed. Sts.': 'Micronesia (Federated States of)',
    'Moldova': 'Republic of Moldova',
    'West Bank and Gaza': 'Palestine',
    'Slovak Republic': 'Slovakia',
    'Tanzania': 'United Republic of Tanzania',
    'Turkiye': 'Turkey',
    'United States': 'United States of America',
    'Venezuela, RB': 'Venezuela (Bolivarian Republic of)',
    'Virgin Islands (U.S.)': 'United States Virgin Islands',
    'Yemen, Rep.': 'Yemen'
}

population_data['Country'] = population_data['Country'].replace(population_name_fixes)

In [10]:
pollutants_countries = set(merged_pollutants_data['Country'].unique())
death_countries = set(all_cause_of_death_data['Country'].unique())
population_countries = set(population_data['Country'].unique()) 

datasets = {
    "Pollutants": pollutants_countries,
    "Death": death_countries,
    "Population": population_countries,
}

report_regions(datasets)

Common in all of them: 197
['Afghanistan', 'Albania', 'Algeria', 'American Samoa', 'Andorra', 'Angola', 'Antigua and Barbuda', 'Argentina', 'Armenia', 'Australia', 'Austria', 'Azerbaijan', 'Bahamas', 'Bahrain', 'Bangladesh', 'Barbados', 'Belarus', 'Belgium', 'Belize', 'Benin', 'Bermuda', 'Bhutan', 'Bolivia (Plurinational State of)', 'Bosnia and Herzegovina', 'Botswana', 'Brazil', 'Brunei Darussalam', 'Bulgaria', 'Burkina Faso', 'Burundi', 'Cabo Verde', 'Cambodia', 'Cameroon', 'Canada', 'Central African Republic', 'Chad', 'Chile', 'China', 'Colombia', 'Comoros', 'Congo', 'Costa Rica', 'Croatia', 'Cuba', 'Cyprus', 'Czechia', "Côte d'Ivoire", "Democratic People's Republic of Korea", 'Democratic Republic of the Congo', 'Denmark', 'Djibouti', 'Dominica', 'Dominican Republic', 'Ecuador', 'Egypt', 'El Salvador', 'Equatorial Guinea', 'Eritrea', 'Estonia', 'Eswatini', 'Ethiopia', 'Fiji', 'Finland', 'France', 'Gabon', 'Gambia', 'Georgia', 'Germany', 'Ghana', 'Greece', 'Greenland', 'Grenada', 'Gu

In [11]:
unique_in_population = population_countries - (pollutants_countries.union(death_countries))
population_data = population_data[~population_data['Country'].isin(unique_in_population)]
print("Remaining countries in population_data:", len(population_data['Country'].unique()))

Remaining countries in population_data: 197


## 5. Comparing Year Coverage Across Datasets

Compare the years available in each dataset to ensure temporal compatibility for merging.

In [12]:

pollutants_years = set(merged_pollutants_data['Year'].unique())
death_years = set(all_cause_of_death_data['Year'].unique())
population_years = set(population_data['Year'].unique())

datasets = {
    "Pollutants": pollutants_years,
    "Death": death_years,
    "Population": population_years,
}

report_regions(datasets)

Common in all of them: 31
[1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]
------------------------------------------------------------


In [13]:
# Export the DataFrame to a CSV file, overwriting the previous file
all_cause_of_death_data.to_csv('all_cause_of_death_data.csv', index=False)
population_data.to_csv('processed_population_data.csv', index=False)

## 7. Merging Datasets

Merge the pollutant, cause of death, and population datasets based on common columns (Country and Year).

In [14]:
# Merge the datasets on 'Country' and 'Year'
merged_data = (
    all_cause_of_death_data
    .merge(merged_pollutants_data, on=['Country', 'Year'], how='left')
    .merge(population_data, on=['Country', 'Year'], how='left')
)

# Display the merged dataset
merged_data.head()

,Country,Cause,Year,Deaths,Deaths Upper,Deaths Lower,Region Name,Exposure Lower HAP,Exposure Mean HAP,Exposure Upper HAP,Exposure Lower NO2,Exposure Mean NO2,Exposure Upper NO2,Exposure Lower OZONE,Exposure Mean OZONE,Exposure Upper OZONE,Exposure Lower PM25,Exposure Mean PM25,Exposure Upper PM25,Population
0,Argentina,Chlamydial infection,1990,2.960181,3.295362,2.695147,Southern Latin America,0.08,0.1,0.11,5.12,18.9,15.2,32.7,33.9,35.2,10.4,19.0,33.4,32755901.0
1,Argentina,Gonococcal infection,1990,1.719927,1.966872,1.512458,Southern Latin America,0.08,0.1,0.11,5.12,18.9,15.2,32.7,33.9,35.2,10.4,19.0,33.4,32755901.0
2,Argentina,Other sexually transmitted infections,1990,10.173681,11.345633,9.253405,Southern Latin America,0.08,0.1,0.11,5.12,18.9,15.2,32.7,33.9,35.2,10.4,19.0,33.4,32755901.0
3,Argentina,Acute hepatitis A,1990,22.093804,45.118908,15.064646,Southern Latin America,0.08,0.1,0.11,5.12,18.9,15.2,32.7,33.9,35.2,10.4,19.0,33.4,32755901.0
4,Argentina,Acute hepatitis B,1990,9.600087,19.328814,5.731842,Southern Latin America,0.08,0.1,0.11,5.12,18.9,15.2,32.7,33.9,35.2,10.4,19.0,33.4,32755901.0


## 8. Final Inspection and Summary

Inspect the merged dataset and summarize the findings regarding the compatibility and quality of the merge operation.

In [15]:
# Shapes of datasets before merging
print("Shape of all_cause_of_death_data:", all_cause_of_death_data.shape)
print("Shape of merged_pollutants_data:", merged_pollutants_data.shape)
print("Shape of population_data:", population_data.shape)

# Shape of the merged dataset
print("Shape of merged_data:", merged_data.shape)

Shape of all_cause_of_death_data: (1460844, 6)
Shape of merged_pollutants_data: (6324, 15)
Shape of population_data: (6107, 3)
Shape of merged_data: (1460844, 20)


In [16]:
merged_data.columns

Index(['Country', 'Cause', 'Year', 'Deaths', 'Deaths Upper', 'Deaths Lower',
       'Region Name', 'Exposure Lower HAP', 'Exposure Mean HAP',
       'Exposure Upper HAP', 'Exposure Lower NO2', 'Exposure Mean NO2',
       'Exposure Upper NO2', 'Exposure Lower OZONE', 'Exposure Mean OZONE',
       'Exposure Upper OZONE', 'Exposure Lower PM25', 'Exposure Mean PM25',
       'Exposure Upper PM25', 'Population'],
      dtype='object')

In [17]:
merged_data.to_csv('FINAL_pollutants_deaths_population.csv', index=False)